# 5. Feature selection: structural conservation <a id="5"></a>
In this section we will featurise the kinase domains in our dataset and select only features of statistical relevance.


## Table of contents

- [5.1 Structural conservation](#24)


## Backend map

How this notebook connects to `workflow/` modules:

```mermaid
flowchart LR
  nb["08a-FeatureSelection"]
  m0["workflow.align_FoldMason"]
  nb --> m0
  m1["workflow.analyse_alignment_foldmason"]
  nb --> m1
  m2["workflow.utilities"]
  nb --> m2
```


![State of the workflow](images/FeatureSelection.png)

In this notebook, we will be focusing on the second step: feature selection.

To get started, let's load some packages!

In [ ]:
from IPython.display import display, HTML

from workflow.align_FoldMason import AlignmentFoldMason
from workflow.analyse_alignment_foldmason import analyse_alignment
from workflow.analyse_alignment_foldmason import visualise_sequence_alignment
from workflow.utilities import PDBDownloader
from workflow.utilities import count_pdb_files, braf_res, clear_and_make, make_seg, copy_filtered_pdbs


## 5.1 Structural conservation  <a id="24"></a>
Here we will be investigating which residues of our reference BRAF kinase are structurally conserved across the collected dataset.


We choose to assess structure conservation using a novel multiple structure alignment algorithm: FoldMason. It uses the structural alphabet from Foldseek to represent 3D structures as sequences, enabling fast comparison between large structure sets. The class `AlignmentFoldMason` is implemented for this purpose.

In [ ]:
from workflow.align_FoldMason import AlignmentFoldMason

# Initialize
aligner = AlignmentFoldMason(log_file="multiple_alignment_foldmason.log")

# Single multi-structure FoldMason run (optionally anchors with the template first)
aligner.process_foldmason_alignment_multi(
    pdb_path="Results/activation_segments/misaligned_filter/",
    target_dir="Results/activation_segments/multi_aligned_foldmason/",
    template_pdb="6UAN_chainD.pdb",  # omit if you don't want to include a template
    out_name="msa",                  # output prefix
    report_mode=1                    # 0: no report, 1: HTML report
)

We will be showing structure conservation with respect to the BRAF reference sequence. We have written the `analyse_alignment()` class to load the multi-structure alignment, calculate conservation at each BRAF residue position and select residues that fall within a certain conservation threshold (70%). We visualise this analysis as a histogram.

**This class creates the `conservation` variable** that is used later in the feature selection workflow.

In order to assess the validity of our approach we can visualise how FoldMason aligns sequences by running the method `visualise_sequence_alignment()`.

In [ ]:
from workflow.analyse_alignment_foldmason import visualise_sequence_alignment

# Create visualizer instance
visualizer = visualise_sequence_alignment()

# Generate HTML for 3Di alignment
di_stats = visualizer.generate_multi_alignment_html(
    alignment_file="Results/activation_segments/multi_aligned_foldmason/msa_3di.fa",
    output_file="Results/multi_alignment_3di.html",
    reference_name="6UAN_chainD"
)

print(f"3Di alignment: {di_stats}")

In [ ]:
from workflow.analyse_alignment_foldmason import analyse_alignment

analyser = analyse_alignment()
conservation_run = analyser.run_multi_alignment_conservation_analysis(
    alignment_file="Results/activation_segments/multi_aligned_foldmason/msa_3di.fa",
    reference_name="6UAN_chainD",
    reference_residues=braf_res(),
    conservation_threshold=0.70,
    output_plot="Results/multi_alignment_foldMason_conservation.png",
    output_csv="Results/conserved_residues_70percent.csv",
    show_plot=True,
)

# Variables used by downstream feature-selection cells
conservation = conservation_run["conservation"]
multi_data = conservation_run["multi_data"]
conserved_df = conservation_run["conserved_df"]